# 03 — Captura RAW de cuatro canales PDM

Preparado en 3.4; ejecución de patrón en placa corresponde a 3.5.
Mantener los micrófonos **desconectados** hasta las instrucciones y comprobaciones de 3.6.
RAW: una palabra uint32 por instante; bits 0..3 = DATA0..3. No convierte a PCM.
Los tiempos UTC son tiempos de ejecución del host; fs=2,4 MHz es nominal.

Este notebook programa la FPGA solo cuando se cambia explícitamente RUN_HARDWARE
al proporcionar el par BIT/HWH y sus hashes aprobados en 3.5. No usar el par de fase 2.


In [ ]:
from pathlib import Path
import hashlib
import json

RUN_HARDWARE = False
BIT_FILE = Path('/home/xilinx/acoustic_camera_phase3_candidate/pdm_raw_capture.bit')
EXPECTED_BIT_SHA256 = '57dff09ba6250d09de3ca4d68e6d83b0304c55a55f7cd903684ff96c93726452'  # Se completa al revisar el candidato en 3.5.
EXPECTED_HWH_SHA256 = 'b794eeaa2a22f1b77aadff4d7485938c31b17e5c3a41ce38eccbb8743d7a9585'
EXPECTED_BUILD_ID = '9FD0DDCDB84D4C52910805339C28CE2E'
EVIDENCE_ROOT = Path('/home/xilinx/acoustic_camera_phase3_evidence')


In [ ]:
if RUN_HARDWARE:
    import re
    for digest in (EXPECTED_BIT_SHA256, EXPECTED_HWH_SHA256):
        if not isinstance(digest, str) or re.fullmatch(r'[0-9a-fA-F]{64}', digest) is None:
            raise ValueError('Introducir los hashes aprobados antes de programar')
    hwh_file = BIT_FILE.with_suffix('.hwh')
    for path, digest in ((BIT_FILE, EXPECTED_BIT_SHA256), (hwh_file, EXPECTED_HWH_SHA256)):
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest.lower():
            raise ValueError('Hash incorrecto: ' + str(path))
    from pynq import Overlay
    from pynq_acoustic_camera import AcousticArray
    overlay = Overlay(str(BIT_FILE), download=False)
    overlay.download()  # Programación explícita; no ejecutar en el paso 3.4.
    array = AcousticArray.from_overlay(overlay, EXPECTED_BUILD_ID, provenance={
        'bit_sha256': EXPECTED_BIT_SHA256, 'hwh_sha256': EXPECTED_HWH_SHA256})
    array.enable_clock()  # Espera CLOCK_READY con plazo.
    print(array.clock.identity)
else:
    print('Captura sin ejecutar; completar candidato y autorización de 3.5.')


## Patrón de transporte

Cinco tamaños y diez repeticiones: orden completo `0..N-1`, longitud y contadores.
Cada resultado contiene una copia de las muestras y metadatos. Cualquier fallo
lanza una excepción y realiza recuperación local; nunca se acepta un bloque parcial.
Los ensayos dirigidos de BUSY/abort/timeout de 3.5 se registrarán junto al candidato.


In [ ]:
if RUN_HARDWARE:
    from datetime import datetime, timezone
    import numpy as np
    run = EVIDENCE_ROOT / datetime.now(timezone.utc).strftime('raw-pattern-%Y%m%dT%H%M%SZ')
    run.mkdir(parents=True, exist_ok=False)
    results = []
    try:
        for n in (1, 2, 17, 4096, 65536):
            for repetition in range(10):
                result = array.capture(n_frames=n, format='raw_pdm', source='pattern')
                np.testing.assert_array_equal(result.samples, np.arange(n, dtype=np.uint32))
                stem = f'pattern-N{n}-rep{repetition:02d}'
                np.save(run / (stem + '.npy'), result.samples)
                (run / (stem + '.json')).write_text(json.dumps(result.metadata, indent=2) + '\n')
                results.append({'n_frames': n, 'repetition': repetition, 'result': 'pass'})
    finally:
        array.disable_clock()
    (run / 'pattern-report.json').write_text(json.dumps(results, indent=2) + '\n')
    print('PASS:', len(results), 'bloques; evidencia:', run)


## Micrófonos y cierre pendientes

No ejecutar SOURCE=microphones antes de 3.6. Tras el cableado verificado, 3.7
comprobará diez bloques físicos de 65536 tramas, los bits reservados, la actividad
de los cuatro canales y los contadores. Esas comprobaciones no sustituyen la
conversión PCM ni la identificación acústica de canales de fases posteriores.
El PDF de cierre será una exportación del BD real de Vivado aceptado.
